# 08_physical_planning_ablation.ipynb – Compact CartPole Planning Ablations

This notebook compares eight small model settings in a synthetic CartPole task. It is an exploratory analogue, not a physical-planning paper reproduction.

Each setting uses the same offline dataset and model seed. The planner uses the same ten evaluation reset and candidate seeds for every setting. The table records each episode return. There is one training seed, so episode variation does not measure training-seed uncertainty.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import gymnasium as gym
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

torch.manual_seed(42)
np.random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

In [ ]:
# Collect one fixed offline dataset. Gymnasium reset and action-space RNGs are separate.
def collect_dataset(episodes=60, gamma=0.99, seed=42):
    env = gym.make("CartPole-v1")
    obs_all, next_all, rew_to_go = [], [], []

    for ep in range(episodes):
        episode_seed = seed + ep
        env.action_space.seed(episode_seed)
        obs, _ = env.reset(seed=episode_seed)
        traj_obs, traj_next, traj_rewards = [], [], []

        done = False
        while not done:
            action = env.action_space.sample()
            nxt, reward, terminated, truncated, _ = env.step(action)
            traj_obs.append(obs.astype(np.float32))
            traj_next.append(nxt.astype(np.float32))
            traj_rewards.append(float(reward))
            obs = nxt
            done = terminated or truncated

        G = 0.0
        returns = []
        for reward in reversed(traj_rewards):
            G = reward + gamma * G
            returns.append(G)
        returns.reverse()
        obs_all.extend(traj_obs)
        next_all.extend(traj_next)
        rew_to_go.extend(returns)

    env.close()
    return (
        torch.tensor(np.array(obs_all), dtype=torch.float32),
        torch.tensor(np.array(next_all), dtype=torch.float32),
        torch.tensor(np.array(rew_to_go), dtype=torch.float32),
    )

obs_t, nxt_t, ret_t = collect_dataset(seed=42)
print(obs_t.shape, nxt_t.shape, ret_t.shape)
print("training reset/action seeds: 42..101")

In [ ]:
# Model factory
def make_mlp(in_dim, hidden_dim, out_dim, depth=2):
    layers = [nn.Linear(in_dim, hidden_dim), nn.ReLU()]
    for _ in range(depth - 1):
        layers += [nn.Linear(hidden_dim, hidden_dim), nn.ReLU()]
    layers += [nn.Linear(hidden_dim, out_dim)]
    return nn.Sequential(*layers)

In [ ]:
# Training + planning evaluator for one ablation setting
def run_setting(latent_dim=64, predictor_depth=2, value_weight=0.1, train_epochs=50, eval_seeds=None, train_seed=42):
    torch.manual_seed(train_seed)
    np.random.seed(train_seed)
    if eval_seeds is None:
        eval_seeds = list(range(200, 210))
    obs_dim = obs_t.shape[1]
    n_actions = 2

    encoder = make_mlp(obs_dim, 128, latent_dim, depth=2).to(device)
    dynamics = make_mlp(latent_dim + n_actions, 128, latent_dim, depth=predictor_depth).to(device)
    value_head = make_mlp(latent_dim, 128, 1, depth=2).to(device)

    optimizer = torch.optim.Adam(
        list(encoder.parameters()) + list(dynamics.parameters()) + list(value_head.parameters()),
        lr=2e-3,
    )

    x = obs_t.to(device)
    y = nxt_t.to(device)
    r = ret_t.to(device).unsqueeze(1)

    for _ in range(train_epochs):
        idx = torch.randperm(x.size(0), device=device)[:512]
        xb, yb, rb = x[idx], y[idx], r[idx]

        z = encoder(xb)
        z_next = encoder(yb).detach()

        # random action one-hot as simple latent transition regularizer
        a_idx = torch.randint(0, n_actions, (xb.size(0),), device=device)
        a_oh = F.one_hot(a_idx, n_actions).float()

        z_pred = dynamics(torch.cat([z, a_oh], dim=-1))
        pred_loss = F.mse_loss(z_pred, z_next)

        v = value_head(z)
        value_loss = F.mse_loss(v, rb)

        loss = pred_loss + value_weight * value_loss
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    # Evaluate the same seeded random continuations on every setting.
    env = gym.make("CartPole-v1")
    ep_rewards = []
    horizon = 10
    n_candidates = 128
    n_rollouts = n_candidates // n_actions

    for episode_seed in eval_seeds:
        episode_seed = int(episode_seed)
        env.action_space.seed(episode_seed)
        np.random.seed(episode_seed)
        obs, _ = env.reset(seed=episode_seed)
        done = False
        total_r = 0.0

        while not done:
            with torch.no_grad():
                obs_tensor = torch.tensor(obs, dtype=torch.float32, device=device).unsqueeze(0)
                z0 = encoder(obs_tensor)
                best_a, best_score = 0, -1e9

                for first_action in range(n_actions):
                    # Vectorized draws preserve the same row-major candidate RNG sequence.
                    continuations = np.random.randint(
                        0, n_actions, size=(n_rollouts, horizon - 1)
                    )
                    action_sequences = np.concatenate(
                        [
                            np.full((n_rollouts, 1), first_action, dtype=np.int64),
                            continuations,
                        ],
                        axis=1,
                    )
                    actions = torch.as_tensor(action_sequences, dtype=torch.long, device=device)
                    z = z0.expand(n_rollouts, -1)
                    for step in range(horizon):
                        action_one_hot = F.one_hot(actions[:, step], n_actions).float()
                        z = dynamics(torch.cat([z, action_one_hot], dim=-1))
                    score = float(value_head(z).mean().item())
                    if score > best_score:
                        best_score = score
                        best_a = first_action

            obs, reward, terminated, truncated, _ = env.step(best_a)
            total_r += reward
            done = terminated or truncated

        ep_rewards.append(total_r)

    env.close()
    return np.array(ep_rewards, dtype=np.float32)

In [ ]:
# Eight settings, one training seed each, and the same ten evaluation seeds.
settings = []
for latent_dim in [32, 64]:
    for predictor_depth in [1, 3]:
        for value_weight in [0.05, 0.2]:
            settings.append((latent_dim, predictor_depth, value_weight))

TRAIN_SEED = 42
EVAL_SEEDS = list(range(200, 210))
rows = []
for latent_dim, predictor_depth, value_weight in tqdm(settings, desc="Ablations"):
    rewards = run_setting(
        latent_dim=latent_dim,
        predictor_depth=predictor_depth,
        value_weight=value_weight,
        train_epochs=40,
        eval_seeds=EVAL_SEEDS,
        train_seed=TRAIN_SEED,
    )
    rows.append(
        {
            "latent_dim": latent_dim,
            "predictor_depth": predictor_depth,
            "value_weight": value_weight,
            "training_seed": TRAIN_SEED,
            "evaluation_seeds": EVAL_SEEDS,
            "episode_rewards": rewards.astype(int).tolist(),
            "reward_mean": float(rewards.mean()),
            "episode_reward_std": float(rewards.std()),
        }
    )

df = pd.DataFrame(rows).sort_values("reward_mean", ascending=False)
print("One training seed per setting; episode standard deviation is not training-seed uncertainty.")
df

In [ ]:
# Visual summary
plt.figure(figsize=(8, 5))
plt.errorbar(
    np.arange(len(df)),
    df["reward_mean"],
    yerr=df["episode_reward_std"],
    fmt="o",
    capsize=4,
    color="tab:blue",
)
plt.xticks(np.arange(len(df)), [f"{ld}/{pd}/{vw}" for ld, pd, vw in zip(df.latent_dim, df.predictor_depth, df.value_weight)], rotation=45, ha="right")
plt.ylabel("Episode reward")
plt.title("Ablation results (latent_dim / depth / value_weight)")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Discussion prompts

1. Which factor gave the largest effect size: latent dimension, dynamics depth, or value loss weight?
2. Do stronger value weights always help, or can they hurt prediction geometry?
3. What would you change to match a real physical benchmark setup more closely?